<a href="https://colab.research.google.com/github/ZiyuCao421/CP5403LLM/blob/main/paper_rag_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Paper RAG Pipeline

一个基于 **FAISS + sentence-transformers** 的论文检索系统。

**数据**: 40 篇 LLM 领域核心论文 (2017–2025)
**模型**: `sentence-transformers/all-MiniLM-L6-v2` (384 维)
**索引**: FAISS (IndexFlatIP, cosine similarity)

## Pipeline

```
Query → Embedding → FAISS Top-k → Prompt → LLM Answer
```

按 cell 顺序运行即可。👇


In [2]:
# Cell 2: 安装依赖（Colab 默认环境通常没有 faiss-cpu / sentence-transformers）
!pip install -q faiss-cpu sentence-transformers transformers torch accelerate bitsandbytes
print("✓ 依赖安装完成")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00
✓ 依赖安装完成


## Step 1-2: 加载数据 (papers.json)

为了避免单 cell 过大被损坏，论文被拆成 **4 段**（每段 10 篇 base64 编码），逐个 cell 运行后合并。

In [3]:
# Cell 3: 加载第 1/4 段论文 (10 篇)
import json, base64
PAPERS_CHUNK_1 = 'W3sidGl0bGUiOiAiQXR0ZW50aW9uIElzIEFsbCBZb3UgTmVlZCIsICJhdXRob3JzIjogIkFzaGlzaCBWYXN3YW5pLCBOb2FtIFNoYXplZXIsIE5pa2kgUGFybWFyLCBKYWtvYiBVc3prb3JlaXQsIExsaW9uIEpvbmVzLCBBaWRhbiBOLiBHb21leiwgTHVrYXN6IEthaXNlciwgSWxsaWEgUG9sb3N1a2hpbiIsICJ5ZWFyIjogMjAxNywgImFic3RyYWN0IjogIlRoZSBkb21pbmFudCBzZXF1ZW5jZSB0cmFuc2R1Y3Rpb24gbW9kZWxzIGFyZSBiYXNlZCBvbiBjb21wbGV4IHJlY3VycmVudCBvciBjb252b2x1dGlvbmFsIG5ldXJhbCBuZXR3b3JrcyBpbiBhbiBlbmNvZGVyLWRlY29kZXIgY29uZmlndXJhdGlvbi4gVGhlIGJlc3QgcGVyZm9ybWluZyBtb2RlbHMgYWxzbyBjb25uZWN0IHRoZSBlbmNvZGVyIGFuZCBkZWNvZGVyIHRocm91Z2ggYW4gYXR0ZW50aW9uIG1lY2hhbmlzbS4gV2UgcHJvcG9zZSBhIG5ldyBzaW1wbGUgbmV0d29yayBhcmNoaXRlY3R1cmUsIHRoZSBUcmFuc2Zvcm1lciwgYmFzZWQgc29sZWx5IG9uIGF0dGVudGlvbiBtZWNoYW5pc21zLCBkaXNwZW5zaW5nIHdpdGggcmVjdXJyZW5jZSBhbmQgY29udm9sdXRpb25zIGVudGlyZWx5LiBFeHBlcmltZW50cyBvbiB0d28gbWFjaGluZSB0cmFuc2xhdGlvbiB0YXNrcyBzaG93IHRoZXNlIG1vZGVscyB0byBiZSBzdXBlcmlvciBpbiBxdWFsaXR5IHdoaWxlIGJlaW5nIG1vcmUgcGFyYWxsZWxpemFibGUgYW5kIHJlcXVpcmluZyBzaWduaWZpY2FudGx5IGxlc3MgdGltZSB0byB0cmFpbi4gT3VyIG1vZGVsIGFjaGlldmVzIDI4LjQgQkxFVSBvbiB0aGUgV01UIDIwMTQgRW5nbGlzaC10by1HZXJtYW4gdHJhbnNsYXRpb24gdGFzaywgaW1wcm92aW5nIG92ZXIgdGhlIGV4aXN0aW5nIGJlc3QgcmVzdWx0cywgaW5jbHVkaW5nIGVuc2VtYmxlcyBieSBvdmVyIDIgQkxFVS4gT24gdGhlIFdNVCAyMDE0IEVuZ2xpc2gtdG8tRnJlbmNoIHRyYW5zbGF0aW9uIHRhc2ssIG91ciBtb2RlbCBlc3RhYmxpc2hlcyBhIG5ldyBzaW5nbGUtbW9kZWwgc3RhdGUtb2YtdGhlLWFydCBCTEVVIHNjb3JlIG9mIDQxLjggYWZ0ZXIgdHJhaW5pbmcgZm9yIDMuNSBkYXlzIG9uIGVpZ2h0IEdQVXMsIGEgc21hbGwgZnJhY3Rpb24gb2YgdGhlIHRyYWluaW5nIGNvc3RzIG9mIHRoZSBiZXN0IG1vZGVscyBmcm9tIHRoZSBsaXRlcmF0dXJlLiBXZSBzaG93IHRoYXQgdGhlIFRyYW5zZm9ybWVyIGdlbmVyYWxpemVzIHdlbGwgdG8gb3RoZXIgdGFza3MgYnkgYXBwbHlpbmcgaXQgc3VjY2Vzc2Z1bGx5IHRvIEVuZ2xpc2ggY29uc3RpdHVlbmN5IHBhcnNpbmcgYm90aCB3aXRoIGxhcmdlIGFuZCBsaW1pdGVkIHRyYWluaW5nIGRhdGEuIn0sIHsidGl0bGUiOiAiQkVSVDogUHJlLXRyYWluaW5nIG9mIERlZXAgQmlkaXJlY3Rpb25hbCBUcmFuc2Zvcm1lcnMgZm9yIExhbmd1YWdlIFVuZGVyc3RhbmRpbmciLCAiYXV0aG9ycyI6ICJKYWNvYiBEZXZsaW4sIE1pbmctV2VpIENoYW5nLCBLZW50b24gTGVlLCBLcmlzdGluYSBUb3V0YW5vdmEiLCAieWVhciI6IDIwMTgsICJhYnN0cmFjdCI6ICJXZSBpbnRyb2R1Y2UgYSBuZXcgbGFuZ3VhZ2UgcmVwcmVzZW50YXRpb24gbW9kZWwgY2FsbGVkIEJFUlQsIHdoaWNoIHN0YW5kcyBmb3IgQmlkaXJlY3Rpb25hbCBFbmNvZGVyIFJlcHJlc2VudGF0aW9ucyBmcm9tIFRyYW5zZm9ybWVycy4gVW5saWtlIHJlY2VudCBsYW5ndWFnZSByZXByZXNlbnRhdGlvbiBtb2RlbHMsIEJFUlQgaXMgZGVzaWduZWQgdG8gcHJlLXRyYWluIGRlZXAgYmlkaXJlY3Rpb25hbCByZXByZXNlbnRhdGlvbnMgZnJvbSB1bmxhYmVsZWQgdGV4dCBieSBqb2ludGx5IGNvbmRpdGlvbmluZyBvbiBib3RoIGxlZnQgYW5kIHJpZ2h0IGNvbnRleHQgaW4gYWxsIGxheWVycy4gQXMgYSByZXN1bHQsIHRoZSBwcmUtdHJhaW5lZCBCRVJUIG1vZGVsIGNhbiBiZSBmaW5lLXR1bmVkIHdpdGgganVzdCBvbmUgYWRkaXRpb25hbCBvdXRwdXQgbGF5ZXIgdG8gY3JlYXRlIHN0YXRlLW9mLXRoZS1hcnQgbW9kZWxzIGZvciBhIHdpZGUgcmFuZ2Ugb2YgdGFza3MsIHN1Y2ggYXMgcXVlc3Rpb24gYW5zd2VyaW5nIGFuZCBsYW5ndWFnZSBpbmZlcmVuY2UsIHdpdGhvdXQgc3Vic3RhbnRpYWwgdGFzay1zcGVjaWZpYyBhcmNoaXRlY3R1cmUgbW9kaWZpY2F0aW9ucy4gQkVSVCBpcyBjb25jZXB0dWFsbHkgc2ltcGxlIGFuZCBlbXBpcmljYWxseSBwb3dlcmZ1bC4gSXQgb2J0YWlucyBuZXcgc3RhdGUtb2YtdGhlLWFydCByZXN1bHRzIG9uIGVsZXZlbiBuYXR1cmFsIGxhbmd1YWdlIHByb2Nlc3NpbmcgdGFza3MsIGluY2x1ZGluZyBwdXNoaW5nIHRoZSBHTFVFIHNjb3JlIHRvIDgwLjUlICg3LjclIHBvaW50IGFic29sdXRlIGltcHJvdmVtZW50KSwgTXVsdGlOTEkgYWNjdXJhY3kgdG8gODYuNyUgKDQuNiUgYWJzb2x1dGUgaW1wcm92ZW1lbnQpLCBTUXVBRCB2MS4xIHF1ZXN0aW9uIGFuc3dlcmluZyBUZXN0IEYxIHRvIDkzLjIgKDEuNSBwb2ludCBhYnNvbHV0ZSBpbXByb3ZlbWVudCkgYW5kIFNRdUFEIHYyLjAgVGVzdCBGMSB0byA4My4xICg1LjEgcG9pbnQgYWJzb2x1dGUgaW1wcm92ZW1lbnQpLiJ9LCB7InRpdGxlIjogIkltcHJvdmluZyBMYW5ndWFnZSBVbmRlcnN0YW5kaW5nIGJ5IEdlbmVyYXRpdmUgUHJlLVRyYWluaW5nIiwgImF1dGhvcnMiOiAiQWxlYyBSYWRmb3JkLCBLYXJ0aGlrIE5hcmFzaW1oYW4sIFRpbSBTYWxpbWFucywgSWx5YSBTdXRza2V2ZXIiLCAieWVhciI6IDIwMTgsICJhYnN0cmFjdCI6ICJXZSBwcm9wb3NlIGEgZ2VuZXJpYyB1bnN1cGVydmlzZWQgcHJlLXRyYWluaW5nIG1ldGhvZCBmb3IgbmF0dXJhbCBsYW5ndWFnZSB1bmRlcnN0YW5kaW5nLiBCeSBwcmUtdHJhaW5pbmcgYSBsYXJnZSBnZW5lcmF0aXZlIGxhbmd1YWdlIG1vZGVsIG9uIGEgbGFyZ2UgZGl2ZXJzZSBjb3JwdXMgb2YgdW5sYWJlbGVkIHRleHQsIGFuZCB0aGVuIGZpbmUtdHVuaW5nIGl0IG9uIGVhY2ggc3BlY2lmaWMgZG93bnN0cmVhbSB0YXNrLCB3ZSBvYnRhaW4gc2lnbmlmaWNhbnQgZ2FpbnMgb3ZlciB0cmFpbmluZyBmcm9tIHNjcmF0Y2ggb24gZGlzY3JpbWluYXRpdmUgdGFza3Mgc3VjaCBhcyBuYXR1cmFsIGxhbmd1YWdlIGluZmVyZW5jZSwgcXVlc3Rpb24gYW5zd2VyaW5nLCBzZW50ZW5jZSBzaW1pbGFyaXR5LCBhbmQgdGV4dCBjbGFzc2lmaWNhdGlvbi4gT3VyIG1vZGVsIGFjaGlldmVzIHJlbWFya2FibGUgcmVzdWx0cyBvbiA5IG9mIHRoZSAxMiB0YXNrcyB3ZSBzdHVkeSwgb2Z0ZW4gb3V0cGVyZm9ybWluZyB0YXNrLXNwZWNpZmljIGFyY2hpdGVjdHVyZXMgYW5kIHN1cnBhc3Npbmcgc3RhdGUtb2YtdGhlLWFydCBieSBhIGxhcmdlIG1hcmdpbi4ifSwgeyJ0aXRsZSI6ICJFeHBsb3JpbmcgdGhlIExpbWl0cyBvZiBUcmFuc2ZlciBMZWFybmluZyB3aXRoIGEgVW5pZmllZCBUZXh0LXRvLVRleHQgVHJhbnNmb3JtZXIiLCAiYXV0aG9ycyI6ICJDb2xpbiBSYWZmZWwsIE5vYW0gU2hhemVlciwgQWRhbSBSb2JlcnRzLCBLYXRoZXJpbmUgTGVlLCBTaGFyYW4gTmFyYW5nLCBNaWNoYWVsIE1hdGVuYSwgWWFucWkgWmhvdSwgV2VpIExpLCBQZXRlciBKLiBMaXUiLCAieWVhciI6IDIwMTksICJhYnN0cmFjdCI6ICJUcmFuc2ZlciBsZWFybmluZywgd2hlcmUgYSBtb2RlbCBpcyBmaXJzdCBwcmUtdHJhaW5lZCBvbiBhIGRhdGEtcmljaCB0YXNrIGJlZm9yZSBiZWluZyBmaW5lLXR1bmVkIG9uIGEgZG93bnN0cmVhbSB0YXNrLCBoYXMgZW1lcmdlZCBhcyBhIHBvd2VyZnVsIHRlY2huaXF1ZSBpbiBuYXR1cmFsIGxhbmd1YWdlIHByb2Nlc3NpbmcgKE5MUCkuIFRoZSBlZmZlY3RpdmVuZXNzIG9mIHRyYW5zZmVyIGxlYXJuaW5nIGhhcyBnaXZlbiByaXNlIHRvIGEgZGl2ZXJzaXR5IG9mIGFwcHJvYWNoZXMsIG1ldGhvZG9sb2d5LCBhbmQgcHJhY3RpY2UuIEluIHRoaXMgcGFwZXIsIHdlIGV4cGxvcmUgdGhlIGxhbmRzY2FwZSBvZiB0cmFuc2ZlciBsZWFybmluZyB0ZWNobmlxdWVzIGZvciBOTFAgYnkgaW50cm9kdWNpbmcgYSB1bmlmaWVkIGZyYW1ld29yayB0aGF0IGNvbnZlcnRzIGFsbCB0ZXh0LWJhc2VkIGxhbmd1YWdlIHByb2JsZW1zIGludG8gYSB0ZXh0LXRvLXRleHQgZm9ybWF0LiBPdXIgc3lzdGVtYXRpYyBzdHVkeSBjb21wYXJlcyBwcmUtdHJhaW5pbmcgb2JqZWN0aXZlcywgYXJjaGl0ZWN0dXJlcywgdW5sYWJlbGVkIGRhdGEgc2V0cywgdHJhbnNmZXIgYXBwcm9hY2hlcywgYW5kIG90aGVyIGZhY3RvcnMgb24gZG96ZW5zIG9mIGxhbmd1YWdlIHVuZGVyc3RhbmRpbmcgdGFza3MuIEJ5IGNvbWJpbmluZyB0aGUgaW5zaWdodHMgZnJvbSBvdXIgZXhwbG9yYXRpb24gd2l0aCBzY2FsZSBhbmQgb3VyIG5ldyBgYENvbG9zc2FsIENsZWFuIENyYXdsZWQgQ29ycHVzJycsIHdlIGFjaGlldmUgc3RhdGUtb2YtdGhlLWFydCByZXN1bHRzIG9uIG1hbnkgYmVuY2htYXJrcyBjb3ZlcmluZyBzdW1tYXJpemF0aW9uLCBxdWVzdGlvbiBhbnN3ZXJpbmcsIHRleHQgY2xhc3NpZmljYXRpb24sIGFuZCBtb3JlLiBUbyBmYWNpbGl0YXRlIGZ1dHVyZSB3b3JrIG9uIHRyYW5zZmVyIGxlYXJuaW5nIGZvciBOTFAsIHdlIHJlbGVhc2Ugb3VyIGRhdGEgc2V0LCBwcmUtdHJhaW5lZCBtb2RlbHMsIGFuZCBjb2RlLiJ9LCB7InRpdGxlIjogIlJvQkVSVGE6IEEgUm9idXN0bHkgT3B0aW1pemVkIEJFUlQgUHJldHJhaW5pbmcgQXBwcm9hY2giLCAiYXV0aG9ycyI6ICJZaW5oYW4gTGl1LCBNeWxlIE90dCwgTmFtYW4gR295YWwsIEppbmdmZWkgRHUsIE1hbmRhciBKb3NoaSwgRGFucWkgQ2hlbiwgT21lciBMZXZ5LCBNaWtlIExld2lzLCBMdWtlIFpldHRsZW1veWVyLCBWZXNlbGluIFN0b3lhbm92IiwgInllYXIiOiAyMDE5LCAiYWJzdHJhY3QiOiAiTGFuZ3VhZ2UgbW9kZWwgcHJldHJhaW5pbmcgaGFzIGxlZCB0byBzaWduaWZpY2FudCBwZXJmb3JtYW5jZSBnYWlucyBidXQgY2FyZWZ1bCBjb21wYXJpc29uIGJldHdlZW4gZGlmZmVyZW50IGFwcHJvYWNoZXMgaXMgY2hhbGxlbmdpbmcuIFRyYWluaW5nIGlzIGNvbXB1dGF0aW9uYWxseSBleHBlbnNpdmUsIG9mdGVuIGRvbmUgb24gcHJpdmF0ZSBkYXRhc2V0cyBvZiBkaWZmZXJlbnQgc2l6ZXMsIGFuZCwgYXMgd2Ugd2lsbCBzaG93LCBoeXBlcnBhcmFtZXRlciBjaG9pY2VzIGhhdmUgc2lnbmlmaWNhbnQgaW1wYWN0IG9uIHRoZSBmaW5hbCByZXN1bHRzLiBXZSBwcmVzZW50IGEgcmVwbGljYXRpb24gc3R1ZHkgb2YgQkVSVCBwcmV0cmFpbmluZyAoRGV2bGluIGV0IGFsLiwgMjAxOSkgdGhhdCBjYXJlZnVsbHkgbWVhc3VyZXMgdGhlIGltcGFjdCBvZiBtYW55IGtleSBoeXBlcnBhcmFtZXRlcnMgYW5kIHRyYWluaW5nIGRhdGEgc2l6ZS4gV2UgZmluZCB0aGF0IEJFUlQgd2FzIHNpZ25pZmljYW50bHkgdW5kZXJ0cmFpbmVkLCBhbmQgY2FuIG1hdGNoIG9yIGV4Y2VlZCB0aGUgcGVyZm9ybWFuY2Ugb2YgZXZlcnkgbW9kZWwgcHVibGlzaGVkIGFmdGVyIGl0LiBPdXIgYmVzdCBtb2RlbCBhY2hpZXZlcyBzdGF0ZS1vZi10aGUtYXJ0IHJlc3VsdHMgb24gR0xVRSwgUkFDRSBhbmQgU1F1QUQuIFRoZXNlIHJlc3VsdHMgaGlnaGxpZ2h0IHRoZSBpbXBvcnRhbmNlIG9mIHByZXZpb3VzbHkgb3Zlcmxvb2tlZCBkZXNpZ24gY2hvaWNlcywgYW5kIHJhaXNlIHF1ZXN0aW9ucyBhYm91dCB0aGUgc291cmNlIG9mIHJlY2VudGx5IHJlcG9ydGVkIGltcHJvdmVtZW50cy4gV2UgcmVsZWFzZSBvdXIgbW9kZWxzIGFuZCBjb2RlLiJ9LCB7InRpdGxlIjogIkxhbmd1YWdlIE1vZGVscyBhcmUgVW5zdXBlcnZpc2VkIE11bHRpdGFzayBMZWFybmVycyIsICJhdXRob3JzIjogIkFsZWMgUmFkZm9yZCwgSmVmZnJleSBXdSwgUmV3b24gQ2hpbGQsIERhdmlkIEx1YW4sIERhcmlvIEFtb2RlaSwgSWx5YSBTdXRza2V2ZXIiLCAieWVhciI6IDIwMTksICJhYnN0cmFjdCI6ICJXZSBkZW1vbnN0cmF0ZSB0aGF0IGxhbmd1YWdlIG1vZGVscyBiZWdpbiB0byBsZWFybiB0YXNrcyB3aXRob3V0IGFueSBleHBsaWNpdCBzdXBlcnZpc2lvbiB3aGVuIHRyYWluZWQgb24gYSBsYXJnZSBkYXRhc2V0IG9mIG1pbGxpb25zIG9mIHdlYnBhZ2VzIGNhbGxlZCBXZWJUZXh0LiBXaGVuIGNvbmRpdGlvbmVkIG9uIGEgZG9jdW1lbnQgcGx1cyBxdWVzdGlvbnMsIHRoZSBhbnN3ZXJzIGdlbmVyYXRlZCBieSB0aGUgbW9kZWwgYXJlIDcxLjIlIGNvcnJlY3Qgb24gdGhlIENvUUEgZGF0YXNldCwgc3VycGFzc2luZyBwcmV2aW91cyBzdGF0ZS1vZi10aGUtYXJ0IGJ5IDE0LjYgcGVyY2VudGFnZSBwb2ludHMuIFRoZSBtb2RlbCBhbHNvIGFjaGlldmVzIGEgbmV3IHN0YXRlIG9mIHRoZSBhcnQgb24gdGhlIFdpbm9ncmFkIFNjaGVtYSBDaGFsbGVuZ2UgYW5kIG90aGVyIHRhc2tzLCBwZXJmb3JtaW5nIGF0IHRoZSBsZXZlbCBvZiByZWNlbnQgYmVuY2htYXJrcyB3aGlsZSBiZWluZyB1bnN1cGVydmlzZWQuIn0sIHsidGl0bGUiOiAiTGFuZ3VhZ2UgTW9kZWxzIGFyZSBGZXctU2hvdCBMZWFybmVycyIsICJhdXRob3JzIjogIlRvbSBCLiBCcm93biwgQmVuamFtaW4gTWFubiwgTmljayBSeWRlciwgTWVsYW5pZSBTdWJiaWFoLCBKYXJlZCBLYXBsYW4sIFByYWZ1bGxhIERoYXJpd2FsLCBBcnZpbmQgTmVlbGFrYW50YW4sIFByYW5hdiBTaHlhbSwgR2lyaXNoIFNhc3RyeSwgQW1hbmRhIEFza2VsbCwgU2FuZGhpbmkgQWdhcndhbCwgQXJpZWwgSGVyYmVydC1Wb3NzLCBHcmV0Y2hlbiBLcnVlZ2VyLCBUb20gSGVuaWdoYW4sIFJld29uIENoaWxkLCBBZGl0eWEgUmFtZXNoLCBEYW5pZWwgTS4gWmllZ2xlciwgSmVmZnJleSBXdSwgQ2xlbWVucyBXaW50ZXIsIENocmlzdG9waGVyIEhlc3NlLCBNYXJrIENoZW4sIEVyaWMgU2lnbGVyLCBNYXRldXN6IExpdHdpbiwgU2NvdHQgR3JheSwgQmVuamFtaW4gQ2hlc3MsIEphY2sgQ2xhcmssIENocmlzdG9waGVyIEJlcm5lciwgU2FtIE1jQ2FuZGxpc2gsIEFsZWMgUmFkZm9yZCwgSWx5YSBTdXRza2V2ZXIsIERhcmlvIEFtb2RlaSIsICJ5ZWFyIjogMjAyMCwgImFic3RyYWN0IjogIlJlY2VudCB3b3JrIGhhcyBkZW1vbnN0cmF0ZWQgc3Vic3RhbnRpYWwgZ2FpbnMgb24gbWFueSBOTFAgdGFza3MgYW5kIGJlbmNobWFya3MgYnkgcHJlLXRyYWluaW5nIG9uIGEgbGFyZ2UgY29ycHVzIG9mIHRleHQgZm9sbG93ZWQgYnkgZmluZS10dW5pbmcgb24gYSBzcGVjaWZpYyB0YXNrLiBXaGlsZSB0eXBpY2FsbHkgdGFzay1hZ25vc3RpYyBpbiBhcmNoaXRlY3R1cmUsIHRoaXMgbWV0aG9kIHN0aWxsIHJlcXVpcmVzIHRhc2stc3BlY2lmaWMgZmluZS10dW5pbmcgZGF0YXNldHMgb2YgdGhvdXNhbmRzIG9yIHRlbnMgb2YgdGhvdXNhbmRzIG9mIGV4YW1wbGVzLiBCeSBjb250cmFzdCwgaHVtYW5zIGNhbiBnZW5lcmFsbHkgcGVyZm9ybSBhIG5ldyBsYW5ndWFnZSB0YXNrIGZyb20gb25seSBhIGZldyBleGFtcGxlcyBvciBmcm9tIHNpbXBsZSBpbnN0cnVjdGlvbnMgLSBzb21ldGhpbmcgd2hpY2ggY3VycmVudCBOTFAgc3lzdGVtcyBzdGlsbCBsYXJnZWx5IHN0cnVnZ2xlIHRvIGRvLiBIZXJlIHdlIHNob3cgdGhhdCBzY2FsaW5nIHVwIGxhbmd1YWdlIG1vZGVscyBncmVhdGx5IGltcHJvdmVzIHRhc2stYWdub3N0aWMsIGZldy1zaG90IHBlcmZvcm1hbmNlLCBzb21ldGltZXMgZXZlbiByZWFjaGluZyBjb21wZXRpdGl2ZW5lc3Mgd2l0aCBwcmlvciBzdGF0ZS1vZi10aGUtYXJ0IGZpbmUtdHVuaW5nIGFwcHJvYWNoZXMuIFNwZWNpZmljYWxseSwgd2UgdHJhaW4gR1BULTMsIGFuIGF1dG9yZWdyZXNzaXZlIGxhbmd1YWdlIG1vZGVsIHdpdGggMTc1IGJpbGxpb24gcGFyYW1ldGVycywgMTB4IG1vcmUgdGhhbiBhbnkgcHJldmlvdXMgbm9uLXNwYXJzZSBsYW5ndWFnZSBtb2RlbCwgYW5kIHRlc3QgaXRzIHBlcmZvcm1hbmNlIGluIHRoZSBmZXctc2hvdCBzZXR0aW5nLiBGb3IgYWxsIHRhc2tzLCBHUFQtMyBpcyBhcHBsaWVkIHdpdGhvdXQgYW55IGdyYWRpZW50IHVwZGF0ZXMgb3IgZmluZS10dW5pbmcsIHdpdGggdGFza3MgYW5kIGZldy1zaG90IGRlbW9uc3RyYXRpb25zIHNwZWNpZmllZCBwdXJlbHkgdmlhIHRleHQgaW50ZXJhY3Rpb24gd2l0aCB0aGUgbW9kZWwuIEdQVC0zIGFjaGlldmVzIHN0cm9uZyBwZXJmb3JtYW5jZSBvbiBtYW55IE5MUCBkYXRhc2V0cywgaW5jbHVkaW5nIHRyYW5zbGF0aW9uLCBxdWVzdGlvbi1hbnN3ZXJpbmcsIGFuZCBjbG96ZSB0YXNrcywgYXMgd2VsbCBhcyBzZXZlcmFsIHRhc2tzIHRoYXQgcmVxdWlyZSBvbi10aGUtZmx5IHJlYXNvbmluZyBvciBkb21haW4gYWRhcHRhdGlvbiwgc3VjaCBhcyB1bnNjcmFtYmxpbmcgd29yZHMsIHVzaW5nIGEgbm92ZWwgd29yZCBpbiBhIHNlbnRlbmNlLCBvciBwZXJmb3JtaW5nIDMtZGlnaXQgYXJpdGhtZXRpYy4gQXQgdGhlIHNhbWUgdGltZSwgd2UgYWxzbyBpZGVudGlmeSBzb21lIGRhdGFzZXRzIHdoZXJlIEdQVC0zJ3MgZmV3LXNob3QgbGVhcm5pbmcgc3RpbGwgc3RydWdnbGVzLCBhcyB3ZWxsIGFzIHNvbWUgZGF0YXNldHMgd2hlcmUgR1BULTMgZmFjZXMgbWV0aG9kb2xvZ2ljYWwgaXNzdWVzIHJlbGF0ZWQgdG8gdHJhaW5pbmcgb24gbGFyZ2Ugd2ViIGNvcnBvcmEuIEZpbmFsbHksIHdlIGZpbmQgdGhhdCBHUFQtMyBjYW4gZ2VuZXJhdGUgc2FtcGxlcyBvZiBuZXdzIGFydGljbGVzIHdoaWNoIGh1bWFuIGV2YWx1YXRvcnMgaGF2ZSBkaWZmaWN1bHR5IGRpc3Rpbmd1aXNoaW5nIGZyb20gYXJ0aWNsZXMgd3JpdHRlbiBieSBodW1hbnMuIFdlIGRpc2N1c3MgYnJvYWRlciBzb2NpZXRhbCBpbXBhY3RzIG9mIHRoaXMgZmluZGluZyBhbmQgb2YgR1BULTMgaW4gZ2VuZXJhbC4ifSwgeyJ0aXRsZSI6ICJSRUFMTTogUmV0cmlldmFsLUF1Z21lbnRlZCBMYW5ndWFnZSBNb2RlbCBQcmUtVHJhaW5pbmciLCAiYXV0aG9ycyI6ICJLZWx2aW4gR3V1LCBLZW50b24gTGVlLCBab3JhIFR1bmcsIFBhbnVwb25nIFBhc3VwYXQsIE1pbmctV2VpIENoYW5nIiwgInllYXIiOiAyMDIwLCAiYWJzdHJhY3QiOiAiTGFuZ3VhZ2UgbW9kZWwgcHJlLXRyYWluaW5nIGhhcyBiZWVuIHNob3duIHRvIGNhcHR1cmUgYSBzdXJwcmlzaW5nIGFtb3VudCBvZiB3b3JsZCBrbm93bGVkZ2UsIGNydWNpYWwgZm9yIE5MUCB0YXNrcyBzdWNoIGFzIHF1ZXN0aW9uIGFuc3dlcmluZy4gSG93ZXZlciwgdGhpcyBrbm93bGVkZ2UgaXMgc3RvcmVkIGltcGxpY2l0bHkgaW4gdGhlIHBhcmFtZXRlcnMgb2YgYSBuZXVyYWwgbmV0d29yaywgcmVxdWlyaW5nIGV2ZXItbGFyZ2VyIG5ldHdvcmtzIHRvIGNvdmVyIG1vcmUgZmFjdHMuIFRvIGNhcHR1cmUga25vd2xlZGdlIGluIGEgbW9yZSBtb2R1bGFyIGFuZCBpbnRlcnByZXRhYmxlIHdheSwgd2UgYXVnbWVudCBsYW5ndWFnZSBtb2RlbCBwcmUtdHJhaW5pbmcgd2l0aCBhIGxhdGVudCBrbm93bGVkZ2UgcmV0cmlldmVyLCB3aGljaCBhbGxvd3MgdGhlIG1vZGVsIHRvIHJldHJpZXZlIGFuZCBhdHRlbmQgb3ZlciBkb2N1bWVudHMgZnJvbSBhIGxhcmdlIGNvcnB1cyBzdWNoIGFzIFdpa2lwZWRpYSwgdXNlZCBkdXJpbmcgcHJlLXRyYWluaW5nLCBmaW5lLXR1bmluZyBhbmQgaW5mZXJlbmNlLiBGb3IgdGhlIGZpcnN0IHRpbWUsIHdlIHNob3cgaG93IHRvIHByZS10cmFpbiBzdWNoIGEga25vd2xlZGdlIHJldHJpZXZlciBpbiBhbiB1bnN1cGVydmlzZWQgbWFubmVyLCB1c2luZyBtYXNrZWQgbGFuZ3VhZ2UgbW9kZWxpbmcgYXMgdGhlIGxlYXJuaW5nIHNpZ25hbCBhbmQgYmFja3Byb3BhZ2F0aW5nIHRocm91Z2ggYSByZXRyaWV2YWwgc3RlcCB0aGF0IGNvbnNpZGVycyBtaWxsaW9ucyBvZiBkb2N1bWVudHMuIFdlIGRlbW9uc3RyYXRlIHRoZSBlZmZlY3RpdmVuZXNzIG9mIFJldHJpZXZhbC1BdWdtZW50ZWQgTGFuZ3VhZ2UgTW9kZWwgcHJlLXRyYWluaW5nIChSRUFMTSkgYnkgZmluZS10dW5pbmcgb24gdGhlIGNoYWxsZW5naW5nIHRhc2sgb2YgT3Blbi1kb21haW4gUXVlc3Rpb24gQW5zd2VyaW5nIChPcGVuLVFBKS4gV2UgY29tcGFyZSBhZ2FpbnN0IHN0YXRlLW9mLXRoZS1hcnQgbW9kZWxzIGZvciBib3RoIGV4cGxpY2l0IGFuZCBpbXBsaWNpdCBrbm93bGVkZ2Ugc3RvcmFnZSBvbiB0aHJlZSBwb3B1bGFyIE9wZW4tUUEgYmVuY2htYXJrcywgYW5kIGZpbmQgdGhhdCB3ZSBvdXRwZXJmb3JtIGFsbCBwcmV2aW91cyBtZXRob2RzIGJ5IGEgc2lnbmlmaWNhbnQgbWFyZ2luICg0LTE2JSBhYnNvbHV0ZSBhY2N1cmFjeSksIHdoaWxlIGFsc28gcHJvdmlkaW5nIHF1YWxpdGF0aXZlIGJlbmVmaXRzIHN1Y2ggYXMgaW50ZXJwcmV0YWJpbGl0eSBhbmQgbW9kdWxhcml0eS4ifSwgeyJ0aXRsZSI6ICJSZXRyaWV2YWwtQXVnbWVudGVkIEdlbmVyYXRpb24gZm9yIEtub3dsZWRnZS1JbnRlbnNpdmUgTkxQIFRhc2tzIiwgImF1dGhvcnMiOiAiUGF0cmljayBMZXdpcywgRXRoYW4gUGVyZXosIEFsZWtzYW5kcmEgUGlrdHVzLCBGYWJpbyBQZXRyb25pLCBWbGFkaW1pciBLYXJwdWtoaW4sIE5hbWFuIEdveWFsLCBIZWlucmljaCBLw7x0dGxlciwgTWlrZSBMZXdpcywgV2VuLXRhdSBZaWgsIFRpbSBSb2NrdMOkc2NoZWwsIFNlYmFzdGlhbiBSaWVkZWwsIERvdXdlIEtpZWxhIiwgInllYXIiOiAyMDIwLCAiYWJzdHJhY3QiOiAiTGFyZ2UgcHJlLXRyYWluZWQgbGFuZ3VhZ2UgbW9kZWxzIGhhdmUgYmVlbiBzaG93biB0byBzdG9yZSBmYWN0dWFsIGtub3dsZWRnZSBpbiB0aGVpciBwYXJhbWV0ZXJzLCBhbmQgYWNoaWV2ZSBzdGF0ZS1vZi10aGUtYXJ0IHJlc3VsdHMgd2hlbiBmaW5lLXR1bmVkIG9uIGRvd25zdHJlYW0gTkxQIHRhc2tzLiBIb3dldmVyLCB0aGVpciBhYmlsaXR5IHRvIGFjY2VzcyBhbmQgcHJlY2lzZWx5IG1hbmlwdWxhdGUga25vd2xlZGdlIGlzIHN0aWxsIGxpbWl0ZWQsIGFuZCBoZW5jZSBvbiBrbm93bGVkZ2UtaW50ZW5zaXZlIHRhc2tzLCB0aGVpciBwZXJmb3JtYW5jZSBsYWdzIGJlaGluZCB0YXNrLXNwZWNpZmljIGFyY2hpdGVjdHVyZXMuIEFkZGl0aW9uYWxseSwgcHJvdmlkaW5nIHByb3ZlbmFuY2UgZm9yIHRoZWlyIGRlY2lzaW9ucyBhbmQgdXBkYXRpbmcgdGhlaXIgd29ybGQga25vd2xlZGdlIHJlbWFpbiBvcGVuIHJlc2VhcmNoIHByb2JsZW1zLiBQcmUtdHJhaW5lZCBtb2RlbHMgd2l0aCBhIGRpZmZlcmVudGlhYmxlIGFjY2VzcyBtZWNoYW5pc20gdG8gZXhwbGljaXQgbm9uLXBhcmFtZXRyaWMgbWVtb3J5IGNhbiBvdmVyY29tZSB0aGlzIGlzc3VlLCBidXQgaGF2ZSBzbyBmYXIgYmVlbiBvbmx5IGludmVzdGlnYXRlZCBmb3IgZXh0cmFjdGl2ZSBkb3duc3RyZWFtIHRhc2tzLiBXZSBleHBsb3JlIGEgZ2VuZXJhbC1wdXJwb3NlIGZpbmUtdHVuaW5nIHJlY2lwZSBmb3IgcmV0cmlldmFsLWF1Z21lbnRlZCBnZW5lcmF0aW9uIChSQUcpIC0tIG1vZGVscyB3aGljaCBjb21iaW5lIHByZS10cmFpbmVkIHBhcmFtZXRyaWMgYW5kIG5vbi1wYXJhbWV0cmljIG1lbW9yeSBmb3IgbGFuZ3VhZ2UgZ2VuZXJhdGlvbi4gV2UgaW50cm9kdWNlIFJBRyBtb2RlbHMgd2hlcmUgdGhlIHBhcmFtZXRyaWMgbWVtb3J5IGlzIGEgcHJlLXRyYWluZWQgc2VxMnNlcSBtb2RlbCBhbmQgdGhlIG5vbi1wYXJhbWV0cmljIG1lbW9yeSBpcyBhIGRlbnNlIHZlY3RvciBpbmRleCBvZiBXaWtpcGVkaWEsIGFjY2Vzc2VkIHdpdGggYSBwcmUtdHJhaW5lZCBuZXVyYWwgcmV0cmlldmVyLiBXZSBjb21wYXJlIHR3byBSQUcgZm9ybXVsYXRpb25zLCBvbmUgd2hpY2ggY29uZGl0aW9ucyBvbiB0aGUgc2FtZSByZXRyaWV2ZWQgcGFzc2FnZXMgYWNyb3NzIHRoZSB3aG9sZSBnZW5lcmF0ZWQgc2VxdWVuY2UsIHRoZSBvdGhlciBjYW4gdXNlIGRpZmZlcmVudCBwYXNzYWdlcyBwZXIgdG9rZW4uIFdlIGZpbmUtdHVuZSBhbmQgZXZhbHVhdGUgb3VyIG1vZGVscyBvbiBhIHdpZGUgcmFuZ2Ugb2Yga25vd2xlZGdlLWludGVuc2l2ZSBOTFAgdGFza3MgYW5kIHNldCB0aGUgc3RhdGUtb2YtdGhlLWFydCBvbiB0aHJlZSBvcGVuIGRvbWFpbiBRQSB0YXNrcywgb3V0cGVyZm9ybWluZyBwYXJhbWV0cmljIHNlcTJzZXEgbW9kZWxzIGFuZCB0YXNrLXNwZWNpZmljIHJldHJpZXZlLWFuZC1leHRyYWN0IGFyY2hpdGVjdHVyZXMuIEZvciBsYW5ndWFnZSBnZW5lcmF0aW9uIHRhc2tzLCB3ZSBmaW5kIHRoYXQgUkFHIG1vZGVscyBnZW5lcmF0ZSBtb3JlIHNwZWNpZmljLCBkaXZlcnNlIGFuZCBmYWN0dWFsIGxhbmd1YWdlIHRoYW4gYSBzdGF0ZS1vZi10aGUtYXJ0IHBhcmFtZXRyaWMtb25seSBzZXEyc2VxIGJhc2VsaW5lLiJ9LCB7InRpdGxlIjogIkFkYXB0ZXJGdXNpb246IE5vbi1EZXN0cnVjdGl2ZSBUYXNrIENvbXBvc2l0aW9uIGZvciBUcmFuc2ZlciBMZWFybmluZyIsICJhdXRob3JzIjogIkpvbmFzIFBmZWlmZmVyLCBBaXNod2FyeWEgS2FtYXRoLCBBbmRyZWFzIFLDvGNrbMOpLCBLeXVuZ2h5dW4gQ2hvLCBJcnluYSBHdXJldnljaCIsICJ5ZWFyIjogMjAyMCwgImFic3RyYWN0IjogIlNlcXVlbnRpYWwgZmluZS10dW5pbmcgYW5kIG11bHRpLXRhc2sgbGVhcm5pbmcgYXJlIG1ldGhvZHMgYWltaW5nIHRvIGluY29ycG9yYXRlIGtub3dsZWRnZSBmcm9tIG11bHRpcGxlIHRhc2tzOyBob3dldmVyLCB0aGV5IHN1ZmZlciBmcm9tIGNhdGFzdHJvcGhpYyBmb3JnZXR0aW5nIGFuZCBkaWZmaWN1bHRpZXMgaW4gZGF0YXNldCBiYWxhbmNpbmcuIFRvIGFkZHJlc3MgdGhlc2Ugc2hvcnRjb21pbmdzLCB3ZSBwcm9wb3NlIEFkYXB0ZXJGdXNpb24sIGEgbmV3IHR3byBzdGFnZSBsZWFybmluZyBhbGdvcml0aG0gdGhhdCBsZXZlcmFnZXMga25vd2xlZGdlIGZyb20gbXVsdGlwbGUgdGFza3MuIEZpcnN0LCBpbiB0aGUga25vd2xlZGdlIGV4dHJhY3Rpb24gc3RhZ2Ugd2UgbGVhcm4gdGFzayBzcGVjaWZpYyBwYXJhbWV0ZXJzIGNhbGxlZCBhZGFwdGVycywgdGhhdCBlbmNhcHN1bGF0ZSB0aGUgdGFzay1zcGVjaWZpYyBpbmZvcm1hdGlvbi4gV2UgdGhlbiBjb21iaW5lIHRoZSBhZGFwdGVycyBpbiBhIHNlcGFyYXRlIGtub3dsZWRnZSBjb21wb3NpdGlvbiBzdGVwLiBXZSBzaG93IHRoYXQgYnkgc2VwYXJhdGluZyB0aGUgdHdvIHN0YWdlcywgaS5lLiwga25vd2xlZGdlIGV4dHJhY3Rpb24gYW5kIGtub3dsZWRnZSBjb21wb3NpdGlvbiwgdGhlIGNsYXNzaWZpZXIgY2FuIGVmZmVjdGl2ZWx5IGV4cGxvaXQgdGhlIHJlcHJlc2VudGF0aW9ucyBsZWFybmVkIGZyb20gbXVsdGlwbGUgdGFza3MgaW4gYSBub24tZGVzdHJ1Y3RpdmUgbWFubmVyLiBXZSBlbXBpcmljYWxseSBldmFsdWF0ZSBBZGFwdGVyRnVzaW9uIG9uIDE2IGRpdmVyc2UgTkxVIHRhc2tzLCBhbmQgZmluZCB0aGF0IGl0IGVmZmVjdGl2ZWx5IGNvbWJpbmVzIHZhcmlvdXMgdHlwZXMgb2Yga25vd2xlZGdlIGF0IGRpZmZlcmVudCBsYXllcnMgb2YgdGhlIG1vZGVsLiBXZSBzaG93IHRoYXQgb3VyIGFwcHJvYWNoIG91dHBlcmZvcm1zIHRyYWRpdGlvbmFsIHN0cmF0ZWdpZXMgc3VjaCBhcyBmdWxsIGZpbmUtdHVuaW5nIGFzIHdlbGwgYXMgbXVsdGktdGFzayBsZWFybmluZy4gT3VyIGNvZGUgYW5kIGFkYXB0ZXJzIGFyZSBhdmFpbGFibGUgYXQgQWRhcHRlckh1Yi5tbC4ifV0='
print(f"✓ 第 1 段：{len(json.loads(base64.b64decode(PAPERS_CHUNK_1).decode('utf-8')))} 篇已加载")

✓ 第 1 段：10 篇已加载


In [4]:
# Cell 4: 加载第 2/4 段论文 (10 篇)
import json, base64
PAPERS_CHUNK_2 = '...'
print(f"✓ 第 2 段：{len(json.loads(base64.b64decode(PAPERS_CHUNK_2).decode('utf-8')))} 篇已加载")

✓ 第 2 段：10 篇已加载


In [5]:
# Cell 5: 加载第 3/4 段论文 (10 篇)
import json, base64
PAPERS_CHUNK_3 = '...'
print(f"✓ 第 3 段：{len(json.loads(base64.b64decode(PAPERS_CHUNK_3).decode('utf-8')))} 篇已加载")

✓ 第 3 段：10 篇已加载


In [6]:
# Cell 6: 加载第 4/4 段论文 (10 篇)
import json, base64
PAPERS_CHUNK_4 = '...'
print(f"✓ 第 4 段：{len(json.loads(base64.b64decode(PAPERS_CHUNK_4).decode('utf-8')))} 篇已加载")

✓ 第 4 段：10 篇已加载


In [7]:
# Cell 7: 合并 4 段 + 解析
import json, base64

PAPERS = []
for i in [1, 2, 3, 4]:
    chunk_var = globals()[f'PAPERS_CHUNK_{i}']
    chunk = json.loads(base64.b64decode(chunk_var).decode('utf-8'))
    PAPERS.extend(chunk)

years = [p['year'] for p in PAPERS]
print(f"✓ 合并完成：{len(PAPERS)} 篇论文")
print(f"  字段: {list(PAPERS[0].keys())}")
print(f"  年份跨度: {min(years)} - {max(years)}")
print(f"  示例: {PAPERS[0]['title']}")

✓ 合并完成：40 篇论文
  字段: ['title', 'authors', 'year', 'abstract']
  年份跨度: 2017 - 2025
  示例: Attention Is All You Need


In [8]:
# Cell 4: 简单 EDA
import pandas as pd
df = pd.DataFrame(PAPERS)
print(df[['year', 'title']].to_string(index=False))
print(f"\nAbstract 长度: min={df['abstract'].str.len().min()}, "
      f"max={df['abstract'].str.len().max()}, "
      f"avg={df['abstract'].str.len().mean():.0f}")

 year                                                                                           title
 2017                                                                       Attention Is All You Need
 2018                BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
 2018                                     Improving Language Understanding by Generative Pre-Training
 2019               Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer
 2019                                         RoBERTa: A Robustly Optimized BERT Pretraining Approach
 2019                                             Language Models are Unsupervised Multitask Learners
 2020                                                           Language Models are Few-Shot Learners
 2020                                          REALM: Retrieval-Augmented Language Model Pre-Training
 2020                                Retrieval-Augmented Generation for Knowledge-

## Step 3: Embedding（Title + Abstract → 384 维向量）

用 `sentence-transformers/all-MiniLM-L6-v2`，把每篇论文的 *Title + Abstract* 拼起来编码。


In [9]:
# Cell 6: 用 sentence-transformers 生成向量
import numpy as np
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

# 拼接 title + abstract
texts = [f"{p['title']}. {p['abstract']}" for p in PAPERS]
print(f"编码 {len(texts)} 篇论文...")

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # 归一化后才能用内积算 cosine
)
print(f"✓ Embeddings shape: {embeddings.shape}")
print(f"  向量维度: {embeddings.shape[1]}")
print(f"  显存占用: {torch.cuda.memory_allocated() / 1024**2:.1f} MB" if device == "cuda" else "")

Device: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

编码 40 篇论文...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✓ Embeddings shape: (40, 384)
  向量维度: 384



In [10]:
# Cell 7: 保存向量到磁盘
np.save("/content/embeddings.npy", embeddings)
print(f"✓ Saved embeddings.npy ({embeddings.nbytes / 1024:.1f} KB)")

✓ Saved embeddings.npy (60.0 KB)


## Step 4: FAISS 索引

用 `IndexFlatIP`（内积），因为我们把 embedding 归一化了，所以内积 ≈ cosine similarity。


In [11]:
# Cell 9: 建立 FAISS 索引
import faiss

dim = embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype("float32"))

print(f"✓ FAISS 索引构建完成")
print(f"  向量数: {index.ntotal}")
print(f"  维度: {dim}")

# 保存
faiss.write_index(index, "/content/papers.index")
print("✓ Saved papers.index")

✓ FAISS 索引构建完成
  向量数: 40
  维度: 384
✓ Saved papers.index


## Step 5-6: 检索函数

输入 query → embed → FAISS Top-k → 返回论文列表。


In [12]:
# Cell 11: 检索函数（retrieve函数已移动到Cell 16，此单元格仅保留测试代码）
# 试一下
test_query = "What paper introduced the Transformer architecture?"
print(f"Q: {test_query}\n")
# retrieve函数已移动到Cell 16，此处不再需要定义和测试
# for r in retrieve(test_query, k=3):
#     print(f"  [{r['score']:.3f}] {r['title']} ({r['year']})")

Q: What paper introduced the Transformer architecture?



## Step 7: Prompt 设计

把检索结果塞进 prompt，让 LLM 按指定格式回答。


In [13]:
# Cell 13: Prompt 模板
PROMPT_TEMPLATE = """You are an academic paper assistant. Based ONLY on the retrieved papers below, answer the user's question in the required format.

## Retrieved Papers
{context}

## User Question
{question}

## Output Format
For each relevant paper, provide:
1. **Title** (exact)
2. **Authors**
3. **Year**
4. **Summary** (1-2 sentences in your own words)
5. **Why Relevant** (explain the connection to the question)

If the papers don't answer the question, say "I don't have enough information from the retrieved papers."
"""


def build_context(retrieved: list[dict]) -> str:
    parts = []
    for i, r in enumerate(retrieved, 1):
        parts.append(
            f"### Paper {i}: {r['title']} ({r['year']})\n"
            f"Authors: {r['authors']}\n"
            f"Abstract: {r['abstract'][:500]}..."
        )
    return "\n\n".join(parts)


def build_prompt(question: str, retrieved: list[dict]) -> str:
    return PROMPT_TEMPLATE.format(
        context=build_context(retrieved),
        question=question,
    )


# 演示：先检查 retrieve 是否存在
question = "What paper introduced the Transformer architecture?"

if "retrieve" in globals() and "model" in globals() and "index" in globals():
    retrieved = retrieve(question, k=3)
    print("=" * 60)
    print("BUILT PROMPT (前 800 字):")
    print(build_prompt(question, retrieved)[:800])
    print("...")
else:
    # 兜底：retrieve 不在时，用 PAPERS 前 3 篇模拟（确保 prompt 模板能展示）
    if "PAPERS" in globals():
        retrieved = []
        for p in PAPERS[:3]:
            retrieved.append({
                "title": p["title"],
                "authors": p["authors"],
                "year": p["year"],
                "abstract": p["abstract"],
                "score": 0.0,
            })
        print("⚠ retrieve 未定义（需要先跑 Cell 3-7 + Cell 10 + Cell 11）")
        print("   用 PAPERS 前 3 篇作为 fallback 演示 prompt 模板：")
        print("=" * 60)
        print("BUILT PROMPT (前 800 字):")
        print(build_prompt(question, retrieved)[:800])
        print("...")
    else:
        print("⚠ 请先按顺序跑 Cell 3-7（加载 PAPERS）→ Cell 10（embedding）→ Cell 11（retrieve）")

⚠ retrieve 未定义（需要先跑 Cell 3-7 + Cell 10 + Cell 11）
   用 PAPERS 前 3 篇作为 fallback 演示 prompt 模板：
BUILT PROMPT (前 800 字):
You are an academic paper assistant. Based ONLY on the retrieved papers below, answer the user's question in the required format.

## Retrieved Papers
### Paper 1: Attention Is All You Need (2017)
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Abstract: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be 
...


## LLM 后端选择

下面提供 3 种方式，按需取消注释其中一行：
- **方式 A (默认)**: 用本地小模型 (Qwen2.5-1.5B-Instruct)，Colab 免费 GPU 跑得动
- **方式 B**: 用 OpenAI / DeepSeek / 任何 OpenAI-compatible API
- **方式 C**: 不调 LLM，直接格式化检索结果（最快）


In [14]:
# Cell 15: LLM 调用封装（支持 3 种后端）

# ============ 方式 A: 本地 HF 模型 (默认) ============
def llm_local(prompt: str, max_new_tokens: int = 400) -> str:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


# ============ 方式 B: OpenAI-compatible API ============
def llm_api(prompt: str, base_url: str, api_key: str, model: str = "gpt-3.5-turbo", max_tokens: int = 400) -> str:
    from openai import OpenAI
    client = OpenAI(base_url=base_url, api_key=api_key)
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.0,
    )
    return resp.choices[0].message.content


# ============ 方式 C: 不调 LLM，直接格式化检索结果 ============
def llm_format_only(retrieved: list[dict], question: str) -> str:
    out = [f"**Question:** {question}\n"]
    for i, r in enumerate(retrieved, 1):
        out.append(f"""---
**Paper {i}**

- **Title**: {r['title']}
- **Authors**: {r['authors']}
- **Year**: {r['year']}
- **Summary**: {r['abstract'][:200]}...
- **Why Relevant**: Score = {r['score']:.3f} in FAISS retrieval for your query.
""")
    return "\n".join(out)


# ============== 选择后端 ==============
# 取消你想用的那一行的注释:

USE_BACKEND = "format"  # 默认 format，最快
# USE_BACKEND = "local"   # 用本地 Qwen2.5-1.5B
# USE_BACKEND = "api"     # 用 API（需要在下方填 key）

# 如果用 API，填这里：
API_CONFIG = {
    "base_url": "https://api.openai.com/v1",  # 或 "https://api.deepseek.com/v1"
    "api_key": "",                            # 你的 key
    "model": "gpt-3.5-turbo",                 # 或 "deepseek-chat"
}

print(f"✓ 后端: {USE_BACKEND}")

✓ 后端: format


In [15]:
# Cell 16: 端到端 RAG pipeline
import numpy as np
import faiss


def retrieve(query: str, k: int = 3):
    q_emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    scores, indices = index.search(q_emb.astype("float32"), k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        p = PAPERS[idx]
        results.append({
            "score": float(score),
            "title": p["title"],
            "authors": p["authors"],
            "year": p["year"],
            "abstract": p["abstract"],
        })
    return results


def rag_answer(question: str, k: int = 3, use_backend: str = None) -> dict:
    if use_backend is None:
        use_backend = globals().get("USE_BACKEND", "format")

    if "retrieve" not in globals() or "model" not in globals() or "index" not in globals():
        missing = [k for k in ["PAPERS", "model", "index", "retrieve"] if k not in globals()]
        raise NameError(f"缺少依赖: {missing}。请先按顺序跑 Cell 3-7 → Cell 10 → Cell 11 → Cell 13。")

    retrieved = retrieve(question, k=k)

    if use_backend == "format":
        # 修 UnboundLocalError: 局部 fallback 用不同名字
        if "llm_format_only" in globals():
            _format_fn = globals()["llm_format_only"]
            answer = _format_fn(retrieved, question)
        else:
            def _format_fallback(retrieved, question):
                out = [f"**Question:** {question}\n"]
                for i, r in enumerate(retrieved, 1):
                    out.append(
                        f"---\n**Paper {i}**\n\n"
                        f"- **Title**: {r['title']}\n"
                        f"- **Authors**: {r['authors']}\n"
                        f"- **Year**: {r['year']}\n"
                        f"- **Summary**: {r['abstract'][:200]}...\n"
                        f"- **Why Relevant**: Score = {r['score']:.3f}"
                    )
                return "\n".join(out)
            answer = _format_fallback(retrieved, question)

    elif use_backend == "local":
        if "build_prompt" not in globals() or "llm_local" not in globals():
            raise NameError("use_backend='local' 需要先跑 Cell 13 + Cell 15")
        prompt = build_prompt(question, retrieved)
        answer = llm_local(prompt)

    elif use_backend == "api":
        cfg = globals().get("API_CONFIG", {})
        if not cfg.get("api_key"):
            answer = "⚠ 请先在 Cell 15 填入 API key"
        else:
            if "build_prompt" not in globals() or "llm_api" not in globals():
                raise NameError("use_backend='api' 需要先跑 Cell 13 + Cell 15")
            prompt = build_prompt(question, retrieved)
            answer = llm_api(prompt, **cfg)

    else:
        answer = f"未知后端: {use_backend}"

    return {"question": question, "retrieved": retrieved, "answer": answer}


# 演示
result = rag_answer("What paper introduced the Transformer architecture?")
print("=" * 70)
print(result["answer"])
print("=" * 70)

**Question:** What paper introduced the Transformer architecture?

---
**Paper 1**

- **Title**: LoRA: Low-Rank Adaptation of Large Language Models
- **Authors**: Edward J. Hu, Yelong Shen, Phillip Wallis, Zeyuan Allen-Zhu, Yuanzhi Li, Shean Wang, Lu Wang, Weizhu Chen
- **Year**: 2021
- **Summary**: An important paradigm of natural language processing consists of large-scale pre-training on general domain data and adaptation to particular tasks or domains. As we pre-train larger models, full fine...
- **Why Relevant**: Score = 0.177 in FAISS retrieval for your query.

---
**Paper 2**

- **Title**: Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer
- **Authors**: Colin Raffel, Noam Shazeer, Adam Roberts, Katherine Lee, Sharan Narang, Michael Matena, Yanqi Zhou, Wei Li, Peter J. Liu
- **Year**: 2019
- **Summary**: Transfer learning, where a model is first pre-trained on a data-rich task before being fine-tuned on a downstream task, has emerged as a powerful

## Step 9: 评估 — 20 个测试问题 + Top-1 / Top-3 准确率

构造 20 个问题，每个问题有 1 个 ground truth 论文。
评估检索器是否能把 ground truth 论文找回来。


In [16]:
# Cell 18: 20 个测试问题 + Ground Truth
TEST_QUERIES = [
    ("What paper introduced the Transformer architecture?", "Attention Is All You Need"),
    ("Which paper proposed BERT for language understanding?", "BERT"),
    ("What is RoBERTa and how does it improve BERT?", "RoBERTa"),
    ("Who wrote the original GPT paper?", "Improving Language Understanding by Generative Pre-Training"),
    ("What is GPT-2?", "Language Models are Unsupervised Multitask Learners"),
    ("Which paper introduced GPT-3?", "Language Models are Few-Shot Learners"),
    ("What is the T5 text-to-text transformer?", "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer"),
    ("What is FLAN?", "Finetuned Language Models Are Zero-Shot Learners"),
    ("Which paper introduced instruction tuning at scale (FLAN-T5)?", "Scaling Instruction-Finetuned Language Models"),
    ("What is Self-Instruct?", "Self-Instruct: Aligning Language Models with Self-Generated Instructions"),
    ("Which paper introduced RLHF for instruction following?", "Training Language Models to Follow Instructions with Human Feedback"),
    ("What is Constitutional AI?", "Constitutional AI: Harmlessness from AI Feedback"),
    ("What is Retrieval-Augmented Generation (RAG)?", "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"),
    ("What is REALM?", "REALM: Retrieval-Augmented Language Model Pre-Training"),
    ("What is Self-RAG?", "Self-RAG: Learning to Retrieve, Generate and Critique through Self-Reflection"),
    ("What is LoRA?", "LoRA: Low-Rank Adaptation of Large Language Models"),
    ("What is QLoRA?", "QLoRA: Efficient Finetuning of Quantized LLMs"),
    ("What is prefix-tuning?", "Prefix-Tuning: Optimizing Continuous Prompts for Generation"),
    ("What is prompt tuning?", "The Power of Scale for Parameter-Efficient Prompt Tuning"),
    ("What is P-Tuning v2?", "P-Tuning v2: Prompt Tuning Can Be Comparable to Fine-Tuning Universally Across Scales and Tasks"),
    ("What is the original LLaMA paper?", "LLaMA: Open and Efficient Foundation Language Models"),
    ("What is Llama 2?", "Llama 2: Open Foundation and Fine-Tuned Chat Models"),
    ("What is Llama 3?", "The Llama 3 Herd of Models"),
    ("What is Alpaca?", "Alpaca: An Instruction-following LLaMA Model"),
    ("What is Vicuna?", "Vicuna: An Open-Source Chatbot Impressing GPT-4"),
    ("What is PaLM?", "PaLM: Scaling Language Modeling with Pathways"),
    ("What is PaLM 2?", "PaLM 2 Technical Report"),
    ("What is Gemini?", "Gemini: A Family of Highly Capable Multimodal Models"),
    ("What is Mistral 7B?", "Mistral 7B"),
    ("What is Mixtral?", "Mixtral of Experts"),
    ("What is Qwen?", "Qwen Technical Report"),
    ("What is Qwen2?", "Qwen2 Technical Report"),
    ("What is DeepSeek-V2?", "DeepSeek-V2 Technical Report"),
    ("What is DeepSeek-R1?", "DeepSeek-R1 Technical Report"),
    ("What is chain-of-thought prompting?", "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models"),
    ("What is Tree of Thoughts?", "Tree of Thoughts: Deliberate Problem Solving with Large Language Models"),
]

print(f"测试集大小: {len(TEST_QUERIES)}")
print(f"示例: {TEST_QUERIES[0]}")

测试集大小: 36
示例: ('What paper introduced the Transformer architecture?', 'Attention Is All You Need')


In [17]:
# Cell 19: 跑评估
import re

def normalize_title(t: str) -> str:
    t = t.lower()
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


top1_correct = 0
top3_correct = 0
all_results = []

for question, ground_truth in TEST_QUERIES:
    retrieved = retrieve(question, k=3)
    retrieved_titles_norm = [normalize_title(r["title"]) for r in retrieved]
    gt_norm = normalize_title(ground_truth)

    # 用子串匹配判断
    top1_hit = gt_norm in retrieved_titles_norm[0] or retrieved_titles_norm[0] in gt_norm
    top3_hit = any(gt_norm in t or t in gt_norm for t in retrieved_titles_norm)

    if top1_hit:
        top1_correct += 1
    if top3_hit:
        top3_correct += 1

    all_results.append({
        "question": question,
        "ground_truth": ground_truth,
        "retrieved_top1": retrieved[0]["title"],
        "retrieved_top3": [r["title"] for r in retrieved],
        "top1_hit": top1_hit,
        "top3_hit": top3_hit,
    })

n = len(TEST_QUERIES)
print("=" * 70)
print(f"📊 评估结果 (共 {n} 个问题)")
print("=" * 70)
print(f"  Top-1 Accuracy: {top1_correct}/{n} = {top1_correct / n * 100:.1f}%")
print(f"  Top-3 Accuracy: {top3_correct}/{n} = {top3_correct / n * 100:.1f}%")
print()

# 列出没命中的
missed = [r for r in all_results if not r["top3_hit"]]
if missed:
    print(f"❌ 未命中 ({len(missed)} 个):")
    for r in missed:
        print(f"  Q: {r['question']}")
        print(f"     GT: {r['ground_truth']}")
        print(f"     Top-1: {r['retrieved_top1']}")
        print()

📊 评估结果 (共 36 个问题)
  Top-1 Accuracy: 23/36 = 63.9%
  Top-3 Accuracy: 28/36 = 77.8%

❌ 未命中 (8 个):
  Q: What paper introduced the Transformer architecture?
     GT: Attention Is All You Need
     Top-1: LoRA: Low-Rank Adaptation of Large Language Models

  Q: Who wrote the original GPT paper?
     GT: Improving Language Understanding by Generative Pre-Training
     Top-1: Self-Instruct: Aligning Language Models with Self-Generated Instructions

  Q: What is GPT-2?
     GT: Language Models are Unsupervised Multitask Learners
     Top-1: Vicuna: An Open-Source Chatbot Impressing GPT-4

  Q: Which paper introduced GPT-3?
     GT: Language Models are Few-Shot Learners
     Top-1: Vicuna: An Open-Source Chatbot Impressing GPT-4

  Q: Which paper introduced RLHF for instruction following?
     GT: Training Language Models to Follow Instructions with Human Feedback
     Top-1: Scaling Instruction-Finetuned Language Models

  Q: What is the original LLaMA paper?
     GT: LLaMA: Open and Efficient

In [18]:
# Cell 20: 详细结果展示
print("=" * 70)
print("📋 详细检索结果")
print("=" * 70)
for i, r in enumerate(all_results, 1):
    mark = "✓" if r["top1_hit"] else ("~" if r["top3_hit"] else "✗")
    print(f"\n[{i:2d}] {mark} Q: {r['question']}")
    print(f"     GT: {r['ground_truth']}")
    print(f"     Top-1: {r['retrieved_top1']}")
    if not r["top1_hit"]:
        print(f"     Top-3: {r['retrieved_top3']}")

📋 详细检索结果

[ 1] ✗ Q: What paper introduced the Transformer architecture?
     GT: Attention Is All You Need
     Top-1: LoRA: Low-Rank Adaptation of Large Language Models
     Top-3: ['LoRA: Low-Rank Adaptation of Large Language Models', 'Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer', 'The Llama 3 Herd of Models']

[ 2] ✓ Q: Which paper proposed BERT for language understanding?
     GT: BERT
     Top-1: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding

[ 3] ✓ Q: What is RoBERTa and how does it improve BERT?
     GT: RoBERTa
     Top-1: RoBERTa: A Robustly Optimized BERT Pretraining Approach

[ 4] ✗ Q: Who wrote the original GPT paper?
     GT: Improving Language Understanding by Generative Pre-Training
     Top-1: Self-Instruct: Aligning Language Models with Self-Generated Instructions
     Top-3: ['Self-Instruct: Aligning Language Models with Self-Generated Instructions', 'LLaMA: Open and Efficient Foundation Language

## Step 10: 完整 Pipeline 可视化

把整个流程 Question → Embedding → FAISS → Retrieved → LLM Summary 走一遍。


In [19]:
# Cell 22: 完整 pipeline 可视化
from IPython.display import display, Markdown

def show_pipeline(question: str, k: int = 3):
    display(Markdown(f"## ❓ Question\n\n> {question}"))

    # Step 1: Embedding
    q_emb = model.encode([question], normalize_embeddings=True, convert_to_numpy=True)
    display(Markdown(f"### 🔢 Step 1: Embedding\n\n- Shape: `{q_emb.shape}`\n- First 5 values: `{q_emb[0][:5].round(3).tolist()}`"))

    # Step 2: FAISS
    scores, indices = index.search(q_emb.astype("float32"), k)
    display(Markdown(f"### 🔍 Step 2: FAISS Top-{k}\n\n- Scores: `{scores[0].round(3).tolist()}`\n- Indices: `{indices[0].tolist()}`"))

    # Step 3: Retrieved Papers
    display(Markdown(f"### 📄 Step 3: Retrieved Papers"))
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        p = PAPERS[idx]
        display(Markdown(
            f"**Rank {rank}** (score={score:.3f})\n\n"
            f"- **Title**: {p['title']}\n"
            f"- **Year**: {p['year']}\n"
            f"- **Authors**: {p['authors'][:80]}...\n"
            f"- **Abstract**: {p['abstract'][:300]}..."
        ))

    # Step 4: LLM Answer
    result = rag_answer(question, k=k)
    display(Markdown(f"### 🤖 Step 4: LLM Answer\n\n{result['answer']}"))


# 跑 3 个示例
for q in [
    "What paper introduced self-attention?",
    "Which paper proposed RLHF?",
    "What is the difference between LoRA and QLoRA?",
]:
    show_pipeline(q)
    print("\n" + "=" * 70 + "\n")

## ❓ Question

> What paper introduced self-attention?

### 🔢 Step 1: Embedding

- Shape: `(1, 384)`
- First 5 values: `[-0.04699999839067459, 0.029999999329447746, -0.03099999949336052, 0.04600000008940697, 0.008999999612569809]`

### 🔍 Step 2: FAISS Top-3

- Scores: `[0.2329999953508377, 0.22699999809265137, 0.21299999952316284]`
- Indices: `[29, 23, 30]`

### 📄 Step 3: Retrieved Papers

**Rank 1** (score=0.233)

- **Title**: Tree of Thoughts: Deliberate Problem Solving with Large Language Models
- **Year**: 2023
- **Authors**: Shunyu Yao, Dian Yu, Jeffrey Zhao, Izhak Shafran, Thomas L. Griffiths, Yuan Cao,...
- **Abstract**: Language models are increasingly being deployed for general problem solving across a wide range of tasks, but are still confined to token-level, left-to-right decision-making processes during inference. This means they can fall short in tasks that require exploration, strategic lookahead, or where i...

...

### 📄 Step 3: Retrieved Papers

**Rank 1** (score=0.277)

- **Title**: QLoRA: Efficient Finetuning of Quantized LLMs
- **Year**: 2023
- **Authors**: Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, Luke Zettlemoyer...
- **Abstract**: We present QLoRA, an efficient finetuning approach that reduces memory usage enough to finetune a 65B parameter model on a single 48GB GPU while preserving full 16-bit finetuning task performance. QLoRA backpropagates gradients through a frozen, 4-bit quantized pretrained language model into Low Ran...

**Rank 2** (score=0.197)

- **Title**: Qwen2 Technical Report
- **Year**: 2024
- **Authors**: An Yang, Baosong Yang, Binyuan Hui, Bo Zheng, Bowen Yu, Chang Zhou, Chengpeng Li...
- **Abstract**: This report introduces the Qwen2 series, the latest addition to our large language models and large multimodal models. We release a comprehensive suite of foundational and instruction-tuned language models, encompassing a parameter range from 0.5 to 72 billion, featuring dense models and a Mixture-o...

**Rank 3** (score=0.164)

- **Title**: LoRA: Low-Rank Adaptation of Large Language Models
- **Year**: 2021
- **Authors**: Edward J. Hu, Yelong Shen, Phillip Wallis, Zeyuan Allen-Zhu, Yuanzhi Li, Shean W...
- **Abstract**: An important paradigm of natural language processing consists of large-scale pre-training on general domain data and adaptation to particular tasks or domains. As we pre-train larger models, full fine-tuning, which retrains all model parameters, becomes less feasible. Using GPT-3 175B as an example ...

### 🤖 Step 4: LLM Answer

**Question:** What is the difference between LoRA and QLoRA?

---
**Paper 1**

- **Title**: QLoRA: Efficient Finetuning of Quantized LLMs
- **Authors**: Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, Luke Zettlemoyer
- **Year**: 2023
- **Summary**: We present QLoRA, an efficient finetuning approach that reduces memory usage enough to finetune a 65B parameter model on a single 48GB GPU while preserving full 16-bit finetuning task performance. QLo...
- **Why Relevant**: Score = 0.277 in FAISS retrieval for your query.

---
**Paper 2**

- **Title**: Qwen2 Technical Report
- **Authors**: An Yang, Baosong Yang, Binyuan Hui, Bo Zheng, Bowen Yu, Chang Zhou, Chengpeng Li, Chengyuan Li, Dayiheng Liu, Fei Huang, Guanting Dong, Haoran Wei, Huan Lin, Jialong Tang, Jialin Wang, Jian Yang, Jianhong Tu, Jianwei Zhang, Jianxin Ma, Jianxin Yang, Jin Xu, Jingren Zhou, Jinze Bai, Jinzheng He, Junyang Lin, Kai Dang, Keming Lu, Keqin Chen, Kexin Yang, Mei Li, Mingfeng Xue, Na Ni, Pei Zhang, Peng Wang, Ru Peng, Rui Men, Ruize Gao, Runji Lin, Shijie Wang, Shuai Bai, Sinan Tan, Tianhang Zhu, Tianhao Li, Tianyu Liu, Wenbin Ge, Xiaodong Deng, Xiaohuan Zhou, Xingzhang Ren, Xinyu Zhang, Xipin Wei, Xuancheng Ren, Xuejing Liu, Yang Fan, Yang Yao, Yichang Zhang, Yu Wan, Yunfei Chu, Yuqiong Liu, Zeyu Cui, Zhenru Zhang, Zhifang Guo, Zhihao Fan
- **Year**: 2024
- **Summary**: This report introduces the Qwen2 series, the latest addition to our large language models and large multimodal models. We release a comprehensive suite of foundational and instruction-tuned language m...
- **Why Relevant**: Score = 0.197 in FAISS retrieval for your query.

---
**Paper 3**

- **Title**: LoRA: Low-Rank Adaptation of Large Language Models
- **Authors**: Edward J. Hu, Yelong Shen, Phillip Wallis, Zeyuan Allen-Zhu, Yuanzhi Li, Shean Wang, Lu Wang, Weizhu Chen
- **Year**: 2021
- **Summary**: An important paradigm of natural language processing consists of large-scale pre-training on general domain data and adaptation to particular tasks or domains. As we pre-train larger models, full fine...
- **Why Relevant**: Score = 0.164 in FAISS retrieval for your query.


## 🎮 试试你自己的问题

修改下面的 `my_question`，然后跑这个 cell。


In [20]:
# Cell 23: 你的自定义问题
my_question = "What is the difference between LLaMA and Llama 2?"  # 改成你想问的
show_pipeline(my_question, k=3)

## ❓ Question

> What is the difference between LLaMA and Llama 2?

### 🔢 Step 1: Embedding

- Shape: `(1, 384)`
- First 5 values: `[-0.0010000000474974513, 0.013000000268220901, -0.04800000041723251, 0.012000000104308128, -0.04399999976158142]`

### 🔍 Step 2: FAISS Top-3

- Scores: `[0.31700000166893005, 0.2849999964237213, 0.24500000476837158]`
- Indices: `[25, 34, 26]`

### 📄 Step 3: Retrieved Papers

**Rank 1** (score=0.317)

- **Title**: Llama 2: Open Foundation and Fine-Tuned Chat Models
- **Year**: 2023
- **Authors**: Hugo Touvron, Louis Martin, Kevin Stone, Peter Albert, Amjad Almahairi, Yasmine ...
- **Abstract**: In this work, we develop and release Llama 2, a collection of pretrained and fine-tuned large language models (LLMs) ranging in scale from 7 billion to 70 billion parameters. Our fine-tuned LLMs, called Llama 2-Chat, are optimized for dialogue use cases. Our models outperform open-source chat models...

**Rank 2** (score=0.285)

- **Title**: The Llama 3 Herd of Models
- **Year**: 2024
- **Authors**: Aaron Grattafiori, Abhimanyu Dubey, Abhinav Jauhri, Abhinav Pandey, Abhishek Kad...
- **Abstract**: Modern artificial intelligence (AI) systems are powered by foundation models. This paper presents a new set of foundation models, called Llama 3. It is a herd of language models that natively support multilinguality, coding, reasoning, and tool usage. Our largest model is a dense Transformer with 40...

**Rank 3** (score=0.245)

- **Title**: Mistral 7B
- **Year**: 2023
- **Authors**: Albert Q. Jiang, Alexandre Sablayrolles, Arthur Mensch, Chris Bamford, Devendra ...
- **Abstract**: We introduce Mistral 7B v0.1, a 7-billion-parameter language model engineered for superior performance and efficiency. Mistral 7B outperforms Llama 2 13B across all evaluated benchmarks, and Llama 1 34B in reasoning, mathematics, and code generation. Our model leverages grouped-query attention (GQA)...

### 🤖 Step 4: LLM Answer

**Question:** What is the difference between LLaMA and Llama 2?

---
**Paper 1**

- **Title**: Llama 2: Open Foundation and Fine-Tuned Chat Models
- **Authors**: Hugo Touvron, Louis Martin, Kevin Stone, Peter Albert, Amjad Almahairi, Yasmine Babaei, Nikolay Bashlykov, Soumya Batra, Prajjwal Bhargava, Shruti Bhosale, Dan Bikel, Lukas Blecher, Cristian Canton Ferrer, Moya Chen, Guillem Cucurull, David Esiobu, Jude Fernandes, Jeremy Fu, Wenyin Fu, Brian Fuller, Cynthia Gao, Vedanuj Goswami, Naman Goyal, Anthony Hartshorn, Saghar Hosseini, Rui Hou, Hakan Inan, Marcin Kardas, Viktor Kerkez, Madian Khabsa, Isabel Kloumann, Artem Korenev, Punit Singh Koura, Marie-Anne Lachaux, Thibaut Lavril, Jenya Lee, Diana Liskovich, Yinghai Lu, Yuning Mao, Xavier Martinet, Todor Mihaylov, Pushkar Mishra, Igor Molybog, Yixin Nie, Andrew Poulton, Jeremy Reizenstein, Rashi Rungta, Kalyan Saladi, Alan Schelten, Ruan Silva, Eric Michael Smith, Ranjan Subramanian, Xiaoqing Ellen Tan, Binh Tang, Ross Taylor, Adina Williams, Jian Xiang Kuan, Puxin Xu, Zheng Yan, Iliyan Zarov, Yuchen Zhang, Angela Fan, Melanie Kambadur, Sharan Narang, Aurelien Rodriguez, Robert Stojnic, Sergey Edunov, Thomas Scialom
- **Year**: 2023
- **Summary**: In this work, we develop and release Llama 2, a collection of pretrained and fine-tuned large language models (LLMs) ranging in scale from 7 billion to 70 billion parameters. Our fine-tuned LLMs, call...
- **Why Relevant**: Score = 0.317 in FAISS retrieval for your query.

---
**Paper 2**

- **Title**: The Llama 3 Herd of Models
- **Authors**: Aaron Grattafiori, Abhimanyu Dubey, Abhinav Jauhri, Abhinav Pandey, Abhishek Kadian, Ahmad Al-Dahle, Aiesha Letman, Akhil Mathur, Alan Schelten, Alex Vaughan, Amy Yang, Angela Fan, Anirudh Goyal, Anthony Hartshorn, Aobo Yang, Archi Mitra, Archie Sravankumar, Artem Korenev, Arthur Hinsvark, Arun Rao, Aston Zhang, Aurelien Rodriguez, Austen Gregerson, Ava Spataru, Baptiste Roziere, Bethany Biron, Binh Tang, Bobbie Chern, Charlotte Caucheteux, Chaya Nayak, Chloe Bi, Chris Marra, Chris McConnell, Christian Keller, Christophe Touret, Chunyang Wu, Corinne Wong, Cristian Canton Ferrer, Cyrus Nikolaidis, Damien Allonsius, Daniel Song, Danielle Pintz, Danny Livshits, Danny Wyatt, David Esiobu, Dhruv Choudhary, Dhruv Mahajan, Diego Garcia-Olano, Diego Perino, Dieuwke Hupkes, Egor Lakomkin, Ehab AlBadawy, Elina Lobanova, Emily Dinan, Eric Michael Smith, Filip Radenovic, Francisco Guzmán, Frank Zhang, Gabriel Synnaeve, Gabrielle Lee, Georgia Lewis Anderson, Govind Thattai, Graeme Nail, Gregoire Mialon, Guan Pang, Guillem Cucurell, Hailey Nguyen, Hannah Korevaar, Hu Xu, Hugo Touvron, Iliyan Zarov, Imanol Arrieta Ibarra, Isabel Kloumann, Ishan Misra, Ivan Evtimov, Jack Zhang, Jade Copet, Jaewon Lee, Jan Geffert, Jana Vranes, Jason Park, Jay Mahadeokar, Jeet Shah, Jelmer van der Linde, Jennifer Billock, Jenny Hong, Jenya Lee, Jeremy Fu, Jianfeng Chi, Jianyu Huang, Jiawen Liu, Jie Wang, Jiecao Yu, Joanna Bitton, Joe Spisak, Jongsoo Park, Joseph Rocca, Joshua Johnstun, Joshua Saxe, Junteng Jia, Kalyan Vasuden Alwala, Karthik Prasad, Kartikeya Upasani, Kate Plawiak, Ke Li, Kenneth Heafield, Kevin Stone, Khalid El-Arini, Krithika Iyer, Kshitiz Malik, Kuenley Chiu, Kunal Bhalla, Kushal Lakhotia, Lauren Rantala-Yeary, Laurens van der Maaten, Lawrence Chen, Liang Tan, Liz Jenkins, Louis Martin, Lovish Madaan, Lubo Malo, Lukas Blecher, Lukas Landzaat, Luke de Oliveira, Madeline Muzzi, Mahesh Pasupuleti, Mannat Singh, Manohar Paluri, Marcin Kardas, Maria Tsimpoukelli, Mathew Oldham, Mathieu Rita, Maya Pavlova, Melanie Kambadur, Mike Lewis, Min Si, Mitesh Kumar Singh, Mona Hassan, Naman Goyal, Narjes Torabi, Nikolay Bashlykov, Nikolay Bogoychev, Niladri Chatterji, Ning Zhang, Olivier Duchenne, Onur Çelebi, Patrick Alrassy, Pengchuan Zhang, Pengwei Li, Petar Vasic, Peter Weng, Prajjwal Bhargava, Pratik Dubal, Praveen Krishnan, Punit Singh Koura, Puxin Xu, Qing He, Qingxiao Dong, Ragavan Srinivasan, Raj Ganapathy, Ramon Calderer, Ricardo Silveira Cabral, Robert Stojnic, Roberta Raileanu, Rohan Maheswari, Rohit Girdhar, Rohit Patel, Romain Sauvestre, Ronnie Polidoro, Roshan Sumbaly, Ross Taylor, Ruan Silva, Rui Hou, Rui Wang, Saghar Hosseini, Sahana Chennabasappa, Sanjay Singh, Sean Bell, Seohyun Sonia Kim, Sergey Edunov, Shaoliang Nie, Sharan Narang, Sharath Raparthy, Sheng Shen, Shengye Wan, Shruti Bhosale, Shun Zhang, Simon Vandenhende, Soumya Batra, Spencer Whitman, Sten Sootla, Stephane Collot, Suchin Gururangan, Sydney Borodinsky, Tamar Herman, Tara Fowler, Tarek Sheasha, Thomas Georgiou, Thomas Scialom, Tobias Speckbacher, Todor Mihaylov, Tong Xiao, Ujjwal Karn, Vedanuj Goswami, Vibhor Gupta, Vignesh Ramanathan, Viktor Kerkez, Vincent Gonguet, Virginie Do, Vish Vogeti, Vítor Albiero, Vladan Petrovic, Weiwei Chu, Wenhan Xiong, Wenyin Fu, Whitney Meers, Xavier Martinet, Xiaodong Wang, Xiaofang Wang, Xiaoqing Ellen Tan, Xide Xia, Xinfeng Xie, Xuchao Jia, Xuewei Wang, Yaelle Goldschlag, Yashesh Gaur, Yasmine Babaei, Yi Wen, Yiwen Song, Yuchen Zhang, Yue Li, Yuning Mao, Zacharie Delpierre Coudert, Zheng Yan, Zhengxing Chen, Zoe Papakipos, Aaditya Singh, Aayushi Srivastava, Abha Jain, Adam Kelsey, Adam Shajnfeld, Adithya Gangidi, Adolfo Victoria, Ahuva Goldstand, Ajay Menon, Ajay Sharma, Alex Boesenberg, Alexei Baevski, Allie Feinstein, Amanda Kallet, Amit Sangani, Amos Teo, Anam Yunus, Andrei Lupu, Andres Alvarado, Andrew Caples, Andrew Gu, Andrew Ho, Andrew Poulton, Andrew Ryan, Ankit Ramchandani, Annie Dong, Annie Franco, Anuj Goyal, Aparajita Saraf, Arkabandhu Chowdhury, Ashley Gabriel, Ashwin Bharambe, Assaf Eisenman, Azadeh Yazdan, Beau James, Ben Maurer, Benjamin Leonhardi, Bernie Huang, Beth Loyd, Beto De Paola, Bhargavi Paranjape, Bing Liu, Bo Wu, Boyu Ni, Braden Hancock, Bram Wasti, Brandon Spence, Brani Stojkovic, Brian Gamido, Britt Montalvo, Carl Parker, Carly Burton, Catalina Mejia, Ce Liu, Changhan Wang, Changkyu Kim, Chao Zhou, Chester Hu, Ching-Hsiang Chu, Chris Cai, Chris Tindal, Christoph Feichtenhofer, Cynthia Gao, Damon Civin, Dana Beaty, Daniel Kreymer, Daniel Li, David Adkins, David Xu, Davide Testuggine, Delia David, Devi Parikh, Diana Liskovich, Didem Foss, Dingkang Wang, Duc Le, Dustin Holland, Edward Dowling, Eissa Jamil, Elaine Montgomery, Eleonora Presani, Emily Hahn, Emily Wood, Eric-Tuan Le, Erik Brinkman, Esteban Arcaute, Evan Dunbar, Evan Smothers, Fei Sun, Felix Kreuk, Feng Tian, Filippos Kokkinos, Firat Ozgenel, Francesco Caggioni, Frank Kanayet, Frank Seide, Gabriela Medina Florez, Gabriella Schwarz, Gada Badeer, Georgia Swee, Gil Halpern, Grant Herman, Grigory Sizov,  Guangyi,  Zhang, Guna Lakshminarayanan, Hakan Inan, Hamid Shojanazeri, Han Zou, Hannah Wang, Hanwen Zha, Haroun Habeeb, Harrison Rudolph, Helen Suk, Henry Aspegren, Hunter Goldman, Hongyuan Zhan, Ibrahim Damlaj, Igor Molybog, Igor Tufanov, Ilias Leontiadis, Irina-Elena Veliche, Itai Gat, Jake Weissman, James Geboski, James Kohli, Janice Lam, Japhet Asher, Jean-Baptiste Gaya, Jeff Marcus, Jeff Tang, Jennifer Chan, Jenny Zhen, Jeremy Reizenstein, Jeremy Teboul, Jessica Zhong, Jian Jin, Jingyi Yang, Joe Cummings, Jon Carvill, Jon Shepard, Jonathan McPhie, Jonathan Torres, Josh Ginsburg, Junjie Wang, Kai Wu, Kam Hou U, Karan Saxena, Kartikay Khandelwal, Katayoun Zand, Kathy Matosich, Kaushik Veeraraghavan, Kelly Michelena, Keqian Li, Kiran Jagadeesh, Kun Huang, Kunal Chawla, Kyle Huang, Lailin Chen, Lakshya Garg, Lavender A, Leandro Silva, Lee Bell, Lei Zhang, Liangpeng Guo, Licheng Yu, Liron Moshkovich, Luca Wehrstedt, Madian Khabsa, Manav Avalani, Manish Bhatt, Martynas Mankus, Matan Hasson, Matthew Lennie, Matthias Reso, Maxim Groshev, Maxim Naumov, Maya Lathi, Meghan Keneally, Miao Liu, Michael L. Seltzer, Michal Valko, Michelle Restrepo, Mihir Patel, Mik Vyatskov, Mikayel Samvelyan, Mike Clark, Mike Macey, Mike Wang, Miquel Jubert Hermoso, Mo Metanat, Mohammad Rastegari, Munish Bansal, Nandhini Santhanam, Natascha Parks, Natasha White, Navyata Bawa, Nayan Singhal, Nick Egebo, Nicolas Usunier, Nikhil Mehta, Nikolay Pavlovich Laptev, Ning Dong, Norman Cheng, Oleg Chernoguz, Olivia Hart, Omkar Salpekar, Ozlem Kalinli, Parkin Kent, Parth Parekh, Paul Saab, Pavan Balaji, Pedro Rittner, Philip Bontrager, Pierre Roux, Piotr Dollar, Polina Zvyagina, Prashant Ratanchandani, Pritish Yuvraj, Qian Liang, Rachad Alao, Rachel Rodriguez, Rafi Ayub, Raghotham Murthy, Raghu Nayani, Rahul Mitra, Rangaprabhu Parthasarathy, Raymond Li, Rebekkah Hogan, Robin Battey, Rocky Wang, Russ Howes, Ruty Rinott, Sachin Mehta, Sachin Siby, Sai Jayesh Bondu, Samyak Datta, Sara Chugh, Sara Hunt, Sargun Dhillon, Sasha Sidorov, Satadru Pan, Saurabh Mahajan, Saurabh Verma, Seiji Yamamoto, Sharadh Ramaswamy, Shaun Lindsay, Shaun Lindsay, Sheng Feng, Shenghao Lin, Shengxin Cindy Zha, Shishir Patil, Shiva Shankar, Shuqiang Zhang, Shuqiang Zhang, Sinong Wang, Sneha Agarwal, Soji Sajuyigbe, Soumith Chintala, Stephanie Max, Stephen Chen, Steve Kehoe, Steve Satterfield, Sudarshan Govindaprasad, Sumit Gupta, Summer Deng, Sungmin Cho, Sunny Virk, Suraj Subramanian, Sy Choudhury, Sydney Goldman, Tal Remez, Tamar Glaser, Tamara Best, Thilo Koehler, Thomas Robinson, Tianhe Li, Tianjun Zhang, Tim Matthews, Timothy Chou, Tzook Shaked, Varun Vontimitta, Victoria Ajayi, Victoria Montanez, Vijai Mohan, Vinay Satish Kumar, Vishal Mangla, Vlad Ionescu, Vlad Poenaru, Vlad Tiberiu Mihailescu, Vladimir Ivanov, Wei Li, Wenchen Wang, Wenwen Jiang, Wes Bouaziz, Will Constable, Xiaocheng Tang, Xiaojian Wu, Xiaolan Wang, Xilun Wu, Xinbo Gao, Yaniv Kleinman, Yanjun Chen, Ye Hu, Ye Jia, Ye Qi, Yenda Li, Yilin Zhang, Ying Zhang, Yossi Adi, Youngjin Nam,  Yu,  Wang, Yu Zhao, Yuchen Hao, Yundi Qian, Yunlu Li, Yuzi He, Zach Rait, Zachary DeVito, Zef Rosnbrick, Zhaoduo Wen, Zhenyu Yang, Zhiwei Zhao, Zhiyu Ma
- **Year**: 2024
- **Summary**: Modern artificial intelligence (AI) systems are powered by foundation models. This paper presents a new set of foundation models, called Llama 3. It is a herd of language models that natively support ...
- **Why Relevant**: Score = 0.285 in FAISS retrieval for your query.

---
**Paper 3**

- **Title**: Mistral 7B
- **Authors**: Albert Q. Jiang, Alexandre Sablayrolles, Arthur Mensch, Chris Bamford, Devendra Singh Chaplot, Diego de las Casas, Florian Bressand, Gianna Lengyel, Guillaume Lample, Lucile Saulnier, Lélio Renard Lavaud, Marie-Anne Lachaux, Pierre Stock, Teven Le Scao, Thibaut Lavril, Thomas Wang, Timothée Lacroix, William El Sayed
- **Year**: 2023
- **Summary**: We introduce Mistral 7B v0.1, a 7-billion-parameter language model engineered for superior performance and efficiency. Mistral 7B outperforms Llama 2 13B across all evaluated benchmarks, and Llama 1 3...
- **Why Relevant**: Score = 0.245 in FAISS retrieval for your query.


## ✅ 完成

- **Step 3**: Embedding (384 维) ✓
- **Step 4**: FAISS Index ✓
- **Step 5-6**: Retrieval ✓
- **Step 7**: Prompt Design ✓
- **Step 9**: 20+ 测试问题 + Top-1/Top-3 准确率 ✓
- **Step 10**: 完整 Pipeline 可视化 ✓

**生成的产物**:
- `/content/embeddings.npy` — 40 × 384 向量
- `/content/papers.index` — FAISS 索引
- 评估结果打印在 Cell 19 / 20
